# Neutron Detection and Moderation

## Introduction

This notebook is your **lab manual**, **analysis notebook**, and **logbook** in one place.

- A notebook consists of **markdown cells** (text) and **code cells** (Python).
- Press **Shift + Enter** to run the current cell.
- Run the notebook **step by step** during the lab rather than all at once.
- If something behaves unexpectedly, try **Kernel → Restart** and then run the notebook again from the top.
- This notebook is **not meant to work fully out of the box**. You will need to fill in values, inspect plots, and make decisions during the analysis.

> **Good practice:** when an error appears, read it carefully. Very often it tells you exactly what is missing.

## Import Python packages

In [ ]:
import sys, os
sys.path.append('./python_files')
import numpy as np  
import matplotlib   
matplotlib.use('Agg') 
%matplotlib ipympl
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
sys.path.append('./python_files')
import MCA, fittingFunctions

# 1. Base measurement

Here you will read in the spectra from the two detectors in their default positions, with **no aluminum** and **no cadmium** inserted.

### Load spectra

In [ ]:
data_detector1 = MCA.load_spectrum("")  # TODO: insert filename for detector 1
data_detector2 = MCA.load_spectrum("")  # TODO: insert filename for detector 2

### Inspect the uncalibrated spectra

Run the next cell to plot the spectra against **channel number**.

> **Note:** the y-axis is logarithmic.

In [ ]:
plt.figure()

plt.step(
    data_detector1.bin_centers,
    data_detector1.counts,
    where="mid",
    color="blue",
    label="Detector 1"
)

plt.step(
    data_detector2.bin_centers,
    data_detector2.counts,
    where="mid",
    color="orange",
    label="Detector 2"
)

plt.title("Detector 1 and Detector 2")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.legend()
plt.show()

### Energy calibration

To convert from **channel number** to **energy**, we assume a linear relation between channel and energy:

<div style="border:1px solid #bbb; border-radius:6px; padding:12px; margin:8px 0;">
<b>Calibration model</b>

$$
E = kC + m
$$

where

- $E$ is the energy,
- $C$ is the channel number,
- $k$ is the slope,
- $m$ is the intercept.
</div>

You will identify three known features in the spectrum for each detector:
- one **peak**,
- two **edges**.

These known energies are then used to determine the calibration coefficients.

### Enter the channel values

Use the uncalibrated spectra to estimate the channel positions of the calibration features.

In [ ]:
# Detector 1
detector1_peak =         # TODO: peak channel
detector1_ch_edge1 =     # TODO: first edge channel
detector1_ch_edge2 =     # TODO: second edge channel

# Detector 2
detector2_peak =         # TODO: peak channel
detector2_ch_edge1 =     # TODO: first edge channel
detector2_ch_edge2 =     # TODO: second edge channel

channels_1 = np.array([detector1_peak, detector1_ch_edge1, detector1_ch_edge2])
channels_2 = np.array([detector2_peak, detector2_ch_edge1, detector2_ch_edge2])

channels_empty = np.row_stack((channels_1, channels_2))
channels_empty

### Enter the corresponding energies

Fill in the known energies that we calculated of the peak and the two edges

**Discussion:** What nuclear reactions do these features correspond to? Why do some reactions produce clear peaks while others appear as edges?

In [ ]:
energy = np.array([
    ,  # TODO: energy of the peak
    ,  # TODO: energy of edge 1
      # TODO: energy of edge 2
])

### Why a straight-line fit?

With more than two calibration points, one line usually cannot pass exactly through every point.  
We therefore use a **least-squares fit**, which finds the line that best matches all points at once.

<div style="border:1px solid #bbb; border-radius:6px; padding:12px; margin:8px 0;">

The fit chooses $k$ and $m$ so that the squared differences between the measured calibration points and the line

$$
E = kC + m
$$

are as small as possible.
</div>

In [ ]:
k_arr = []
m_arr = []

def line_func(x, k, m):
    return k * x + m

for i, channels in enumerate(channels_empty):
    initial_guess = [2, 1]

    estimates, covariance_matrix = curve_fit(
        line_func,
        channels,
        energy,
        p0=initial_guess
    )

    k_arr.append(estimates[0])
    m_arr.append(estimates[1])

    print(f"Detector {i+1}: k = {k_arr[i]:.4f}, m = {m_arr[i]:.4f}")

    plt.figure()
    plt.plot(channels, energy, "*", label="Calibration points")
    plt.plot(channels, line_func(channels, k_arr[i], m_arr[i]), label="Linear fit")
    plt.title(f"Energy calibration for Detector {i+1}")
    plt.xlabel("Channel")
    plt.ylabel("Energy [keV]")
    plt.legend()
    plt.show()

### Apply the calibration

In [ ]:
data_detector1.calibrate(k_arr[0], m_arr[0])
data_detector2.calibrate(k_arr[1], m_arr[1])

### Plot calibrated spectra

In [ ]:
plt.figure()
plt.step(data_detector1.energy, data_detector1.counts, where="mid", label="Detector 1")
plt.title("Calibrated spectrum: Detector 1")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.legend()

plt.figure()
plt.step(data_detector2.energy, data_detector2.counts, where="mid", label="Detector 2")
plt.title("Calibrated spectrum: Detector 2")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.legend()

## Discussion

- Identify the main regions in each calibrated spectrum.
- Which features correspond to full-energy deposition?
- Which features correspond to partial energy deposition or escape processes?

# 2. Measurement with aluminum

Now load the spectra measured after inserting the **aluminum sheet**.

### Load spectra

In [ ]:
data_detector1_aluminum = MCA.load_spectrum("")  # TODO: detector 1 with aluminum
data_detector2_aluminum = MCA.load_spectrum("")  # TODO: detector 2 with aluminum

### Inspect the uncalibrated spectra

In [ ]:
plt.figure()
plt.step(data_detector1_aluminum.bin_centers, data_detector1_aluminum.counts, where="mid")
plt.title("Detector 1 with aluminum")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")

plt.figure()
plt.step(data_detector2_aluminum.bin_centers, data_detector2_aluminum.counts, where="mid")
plt.title("Detector 2 with aluminum")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.show()

### Apply the previous calibration

The detector electronics have not changed, so you can reuse the calibration coefficients found above.

In [ ]:
data_detector1_aluminum.calibrate(k_arr[0], m_arr[0])
data_detector2_aluminum.calibrate(k_arr[1], m_arr[1])

### Compare Detector 1 with and without aluminum

Before comparing the spectra, normalize the counts by the measurement duration.

In [ ]:
counts_detector1_norm = data_detector1.counts / data_detector1.duration
counts_detector1_al_norm = data_detector1_aluminum.counts / data_detector1_aluminum.duration

In [ ]:
plt.figure()
plt.step(data_detector1.energy, counts_detector1_norm, where="mid", label="Detector 1")
plt.step(data_detector1_aluminum.energy, counts_detector1_al_norm, where="mid", label="Detector 1 with aluminum")
plt.title("Detector 1: with and without aluminum")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts")
plt.legend(loc="upper right")
plt.show()

### Ratio for Detector 1

A ratio plot is often easier to interpret than two overlaid spectra.

> **Tip:** when choosing an energy interval for the average ratio, look for a region where the ratio is reasonably flat and not dominated by very low statistics.

In [ ]:
E_cut = 1500

N_bins = len(data_detector1.energy)
ratio_al_det1 = np.zeros(N_bins)

for i in range(N_bins):
    if (
        counts_detector1_norm[i] != 0
        and counts_detector1_al_norm[i] != 0
        and data_detector1.energy[i] < E_cut
    ):
        ratio_al_det1[i] = counts_detector1_al_norm[i] / counts_detector1_norm[i]

In [ ]:
plt.figure()
plt.step(data_detector1.energy, counts_detector1_norm, where="mid", label="Detector 1")
plt.step(data_detector1_aluminum.energy, counts_detector1_al_norm, where="mid", label="Detector 1 with aluminum")
plt.step(data_detector1_aluminum.energy, ratio_al_det1, where="mid", label="Ratio")
plt.title("Detector 1 normalized spectra and ratio")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts / ratio (log scale)")
plt.yscale("log")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_min =    # TODO: choose a suitable lower energy limit
E_max =    # TODO: choose a suitable upper energy limit

ratio_sum = 0
ratio_N = 0

for i in range(N_bins):
    if data_detector1.energy[i] >= E_min and data_detector1.energy[i] <= E_max:
        ratio_sum += ratio_al_det1[i]
        ratio_N += 1

ratio_avg_al_1 = ratio_sum / ratio_N
print(f"Average ratio with / without aluminum in [{E_min}, {E_max}] keV = {ratio_avg_al_1:.3f}")

### Compare Detector 2 with and without aluminum

In [ ]:
counts_detector2_norm = data_detector2.counts / data_detector2.duration
counts_detector2_al_norm = data_detector2_aluminum.counts / data_detector2_aluminum.duration

In [ ]:
plt.figure()
plt.step(data_detector2.energy, counts_detector2_norm, where="mid", label="Detector 2")
plt.step(data_detector2_aluminum.energy, counts_detector2_al_norm, where="mid", label="Detector 2 with aluminum")
plt.title("Detector 2: with and without aluminum")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_cut = 1500

N_bins = len(data_detector2.energy)
ratio_al_det2 = np.zeros(N_bins)

for i in range(N_bins):
    if (
        counts_detector2_norm[i] != 0
        and counts_detector2_al_norm[i] != 0
        and data_detector2.energy[i] < E_cut
    ):
        ratio_al_det2[i] = counts_detector2_al_norm[i] / counts_detector2_norm[i]

In [ ]:
plt.figure()
plt.step(data_detector2.energy, counts_detector2_norm, where="mid", label="Detector 2")
plt.step(data_detector2_aluminum.energy, counts_detector2_al_norm, where="mid", label="Detector 2 with aluminum")
plt.step(data_detector2_aluminum.energy, ratio_al_det2, where="mid", label="Ratio")
plt.title("Detector 2 normalized spectra and ratio")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts / ratio")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_min =    # TODO: choose a suitable lower energy limit
E_max =    # TODO: choose a suitable upper energy limit

ratio_sum = 0
ratio_N = 0

for i in range(N_bins):
    if data_detector2.energy[i] >= E_min and data_detector2.energy[i] <= E_max:
        ratio_sum += ratio_al_det2[i]
        ratio_N += 1

ratio_avg_al_2 = ratio_sum / ratio_N
print(f"Average ratio with / without aluminum in [{E_min}, {E_max}] keV = {ratio_avg_al_2:.3f}")

## Discussion

- What effect does the aluminum foil have on Detector 1?
- What effect does the aluminum foil have on Detector 2?
- Are the changes energy dependent?
- What does the ratio of the two measurements represent physically?

# 3. Measurement with cadmium

Now load the spectra measured after inserting the **cadmium sheet**.

### Load spectra

In [ ]:
data_detector1_cadmium = MCA.load_spectrum("")  # TODO: detector 1 with cadmium
data_detector2_cadmium = MCA.load_spectrum("")  # TODO: detector 2 with cadmium

In [ ]:
plt.figure()
plt.step(data_detector1_cadmium.bin_centers, data_detector1_cadmium.counts, where="mid")
plt.title("Detector 1 with cadmium")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")

plt.figure()
plt.step(data_detector2_cadmium.bin_centers, data_detector2_cadmium.counts, where="mid")
plt.title("Detector 2 with cadmium")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.show()

### Apply the previous calibration

In [ ]:
data_detector1_cadmium.calibrate(k_arr[0], m_arr[0])
data_detector2_cadmium.calibrate(k_arr[1], m_arr[1])

### Compare Detector 1 with and without cadmium

In [ ]:
counts_detector1_cd_norm = data_detector1_cadmium.counts / data_detector1_cadmium.duration

In [ ]:
plt.figure()
plt.step(data_detector1.energy, counts_detector1_norm, where="mid", label="Detector 1")
plt.step(data_detector1_cadmium.energy, counts_detector1_cd_norm, where="mid", label="Detector 1 with cadmium")
plt.title("Detector 1: with and without cadmium")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_cut = 1500

N_bins = len(data_detector1.energy)
ratio_cd_det1 = np.zeros(N_bins)

for i in range(N_bins):
    if (
        counts_detector1_norm[i] != 0
        and counts_detector1_cd_norm[i] != 0
        and data_detector1.energy[i] < E_cut
    ):
        ratio_cd_det1[i] = counts_detector1_cd_norm[i] / counts_detector1_norm[i]

In [ ]:
plt.figure()
plt.step(data_detector1.energy, counts_detector1_norm, where="mid", label="Detector 1")
plt.step(data_detector1_cadmium.energy, counts_detector1_cd_norm, where="mid", label="Detector 1 with cadmium")
plt.step(data_detector1_cadmium.energy, ratio_cd_det1, where="mid", label="Ratio")
plt.title("Detector 1 normalized spectra and ratio")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts / ratio")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_min =    # TODO: choose a suitable lower energy limit
E_max =    # TODO: choose a suitable upper energy limit

ratio_sum = 0
ratio_N = 0

for i in range(N_bins):
    if data_detector1.energy[i] >= E_min and data_detector1.energy[i] <= E_max:
        ratio_sum += ratio_cd_det1[i]
        ratio_N += 1

ratio_avg_cd_1 = ratio_sum / ratio_N
print(f"Average ratio with / without cadmium in [{E_min}, {E_max}] keV = {ratio_avg_cd_1:.3f}")

### Compare Detector 2 with and without cadmium

In [ ]:
counts_detector2_cd_norm = data_detector2_cadmium.counts / data_detector2_cadmium.duration

In [ ]:
plt.figure()
plt.step(data_detector2.energy, counts_detector2_norm, where="mid", label="Detector 2")
plt.step(data_detector2_cadmium.energy, counts_detector2_cd_norm, where="mid", label="Detector 2 with cadmium")
plt.title("Detector 2: with and without cadmium")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_cut = 1500

N_bins = len(data_detector2.energy)
ratio_cd_det2 = np.zeros(N_bins)

for i in range(N_bins):
    if (
        counts_detector2_norm[i] != 0
        and counts_detector2_cd_norm[i] != 0
        and data_detector2.energy[i] < E_cut
    ):
        ratio_cd_det2[i] = counts_detector2_cd_norm[i] / counts_detector2_norm[i]

In [ ]:
plt.figure()
plt.step(data_detector2.energy, counts_detector2_norm, where="mid", label="Detector 2")
plt.step(data_detector2_cadmium.energy, counts_detector2_cd_norm, where="mid", label="Detector 2 with cadmium")
plt.step(data_detector2_cadmium.energy, ratio_cd_det2, where="mid", label="Ratio")
plt.title("Detector 2 normalized spectra and ratio")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts / ratio")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_min =    # TODO: choose a suitable lower energy limit
E_max =    # TODO: choose a suitable upper energy limit

ratio_sum = 0
ratio_N = 0

for i in range(N_bins):
    if data_detector2.energy[i] >= E_min and data_detector2.energy[i] <= E_max:
        ratio_sum += ratio_cd_det2[i]
        ratio_N += 1

ratio_avg_cd_2 = ratio_sum / ratio_N
print(f"Average ratio with / without cadmium in [{E_min}, {E_max}] keV = {ratio_avg_cd_2:.3f}")

## Discussion

- What effect does the cadmium sheet have on Detector 1?
- What effect does the cadmium sheet have on Detector 2?
- Which parts of the spectrum are most affected, and why?

# 4. Change of detector position

In this part, Detector 2 is moved to a second position and compared with its original position.

### Load spectrum

In [ ]:
data_detector2_position2 = MCA.load_spectrum("")  # TODO: insert filename

In [ ]:
plt.figure()
plt.step(data_detector2_position2.bin_centers, data_detector2_position2.counts, where="mid")
plt.title("Detector 2 in position 2")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.show()

### Apply the previous calibration

In [ ]:
data_detector2_position2.calibrate(k_arr[1], m_arr[1])

### Compare position 1 and position 2

In [ ]:
counts_detector2_pos2_norm = data_detector2_position2.counts / data_detector2_position2.duration

In [ ]:
plt.figure()
plt.step(data_detector2.energy, counts_detector2_norm, where="mid", label="Position 1")
plt.step(data_detector2_position2.energy, counts_detector2_pos2_norm, where="mid", label="Position 2")
plt.title("Detector 2 in two positions")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_cut = 1500

N_bins = len(data_detector2.energy)
ratio_det2_pos = np.zeros(N_bins)

for i in range(N_bins):
    if (
        counts_detector2_norm[i] != 0
        and counts_detector2_pos2_norm[i] != 0
        and data_detector2.energy[i] < E_cut
    ):
        ratio_det2_pos[i] = counts_detector2_pos2_norm[i] / counts_detector2_norm[i]

In [ ]:
plt.figure()
plt.step(data_detector2.energy, counts_detector2_norm, where="mid", label="Position 1")
plt.step(data_detector2_position2.energy, counts_detector2_pos2_norm, where="mid", label="Position 2")
plt.step(data_detector2_position2.energy, ratio_det2_pos, where="mid", label="Ratio")
plt.title("Detector 2 in two positions and their ratio")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Normalized counts / ratio")
plt.legend(loc="upper right")
plt.show()

In [ ]:
E_min =    # TODO: choose a suitable lower energy limit
E_max =    # TODO: choose a suitable upper energy limit

ratio_sum = 0
ratio_N = 0

for i in range(N_bins):
    if data_detector2.energy[i] >= E_min and data_detector2.energy[i] <= E_max:
        ratio_sum += ratio_det2_pos[i]
        ratio_N += 1

ratio_avg_pos = ratio_sum / ratio_N
print(f"Average ratio between position 1 and position 2 in [{E_min}, {E_max}] keV = {ratio_avg_pos:.3f}")

## Discussion

- How does increasing the distance affect the count rate?
- Does the change agree with the inverse-square law?
- If not exactly, what experimental effects could explain the difference?

# 5. BGO spectrum

In this part, you will inspect and calibrate a BGO gamma spectrum.

### Load spectrum

In [ ]:
data_BGO = MCA.load_spectrum("")  # TODO: insert filename

In [ ]:
plt.figure()
plt.step(data_BGO.bin_centers, data_BGO.counts, where="mid")
plt.title("Uncalibrated BGO gamma spectrum")
plt.xlabel("Channel")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.show()

### Apply the calibration

Use the calibration coefficients given in the `.Spe` file under `$ENER_FIT`.

In [ ]:
data_BGO.calibrate( , )  # TODO: insert the two calibration coefficients

In [ ]:
plt.figure()
plt.step(data_BGO.energy, data_BGO.counts, where="mid")
plt.title("Calibrated BGO gamma spectrum")
plt.xlabel("Energy [keV]")
plt.ylabel("Counts (log scale)")
plt.yscale("log")
plt.show()

## Discussion

- Where do the gamma rays come from?
- What is the energy of the broad feature near 2.2 MeV?
- What are the smaller peaks to the left of it?

# 6. Estimate the branching ratio of the $^{10}\mathrm{B}(n,\alpha){}^{7}\mathrm{Li}$ reaction

**This part is mandatory for FYSC22 only.**

### Sum spectra

For this task, sum the Detector 1 spectra from:
- the base measurement,
- the measurement with aluminum,
- the measurement with cadmium.

In [ ]:
counts_sum = (
    data_detector1.counts
    + data_detector1_aluminum.counts
    + data_detector1_cadmium.counts
)

In [ ]:
plt.figure()
plt.step(data_detector1.energy, data_detector1.counts, where="mid", label="Base measurement")
plt.step(data_detector1.energy, counts_sum, where="mid", label="Summed spectra")
plt.title("Detector 1 summed statistics")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Counts")
plt.legend(loc="upper right")
plt.show()


### Excited-state branch

$$
n + {}^{10}\mathrm{B}
\rightarrow
\alpha + {}^{7}\mathrm{Li}^{*}
$$


### Ground-state branch

$$
n + {}^{10}\mathrm{B}
\rightarrow
\alpha + {}^{7}\mathrm{Li}
$$
Choose an energy interval and construct approximate regions for the total population and the ground-state population.

> **Tip:**  - Go through the code to get an understanding first about what values you need to fill in and try out different values.

We are NOT perfectly separating the two physical branches.
Instead, we make a rough graphical estimate of two areas:

  1) area_all_branches:
     approximate area representing all boron events

  2) area_ground_branch:
     approximate area representing the ground-state branch

Then:

  ground-state fraction = area_ground_branch / area_all_branches


In [ ]:
# Measured spectrum
energy_keV = data_detector1.energy       # x-axis: deposited energy [keV]
counts = data_detector1.counts           # y-axis: measured counts


# Energy interval included in the area estimate:
estimate_min_keV =  ### ToDo : choose a good starting energy from the plot above 
estimate_max_keV =  ### ToDo : choose a good end point from the plot above 

# ------------------------------------------------------------
# Replace the measured counts by a flat count level. 
# ------------------------------------------------------------

all_branches_flat_below_keV =  ### ToDo: Select the max E for which the measured counts are replaced by a flat count level. 
all_branches_flat_counts =   ### ToDo: Select counts value value with which to replace/

#Ground state only
ground_branch_flat_below_keV =  ### ToDo: Select the max E for which the measured counts are replaced by a flat count level.
ground_branch_flat_counts = ### ToDo: Select counts value value with which to replace.


# ------------------------------------------------------------
# Create arrays for the two estimated curves
# ------------------------------------------------------------

all_branches_estimate = np.zeros_like(counts)
ground_branch_estimate = np.zeros_like(counts)


# ------------------------------------------------------------
# Fill the estimated curves (flat count level + the measured counts )
# ------------------------------------------------------------

for i, E in enumerate(energy_keV):

    # Only include the chosen energy range
    if E < estimate_min_keV or E > estimate_max_keV:
        continue

    # Estimate for all boron branches combined
    if E < all_branches_flat_below_keV:
        all_branches_estimate[i] = all_branches_flat_counts
    else:
        all_branches_estimate[i] = counts[i]

    # Estimate for the ground-state branch
    if E < ground_branch_flat_below_keV:
        ground_branch_estimate[i] = ground_branch_flat_counts
    else:
        ground_branch_estimate[i] = counts[i]


# ------------------------------------------------------------
# Calculate areas
# ------------------------------------------------------------

# Since the energy bin width is constant, summing counts is proportional
# to the area under each curve.

area_all_branches = np.sum(all_branches_estimate)
area_ground_branch = np.sum(ground_branch_estimate)

ground_state_fraction = area_ground_branch / area_all_branches


print(f"Estimated area for all boron branches:   {area_all_branches:.0f}")
print(f"Estimated area for ground-state branch:  {area_ground_branch:.0f}")
print(f"Estimated ground-state fraction:         {ground_state_fraction:.3f}")
print(f"Estimated ground-state percentage:       {100*ground_state_fraction:.1f}%")

In [ ]:
plt.figure(figsize=(9, 6))

# Measured spectrum
plt.step(
    energy_keV,
    counts,
    where="mid",
    color="black",
    linewidth=1.5,
    alpha=0.20,
    label="Measured spectrum",
)

# Estimated area for all boron branches
plt.step(
    energy_keV,
    all_branches_estimate,
    where="mid",
    color="tab:orange",
    linewidth=2,
    alpha=0.20,
    label="Estimated area: all boron branches",
)


# Estimated area for ground-state branch
plt.step(
    energy_keV,
    ground_branch_estimate,
    where="mid",
    color="tab:green",
    linewidth=2,
    alpha=0.40,
    label="Estimated area: ground-state branch",
)


plt.fill_between(
    energy_keV,
    ground_branch_estimate,
    step="mid",
    color="tab:green",
    alpha=0.2,
)

# Physical endpoints
plt.axvline(
    1470,
    color="tab:orange",
    linestyle="--",
    linewidth=1.5,
    label="Excited-state endpoint ~1470 keV",
)

plt.axvline(
    1780,
    color="tab:green",
    linestyle="--",
    linewidth=1.5,
    label="Ground-state endpoint ~1780 keV",
)


plt.fill_between(
    energy_keV,
    all_branches_estimate,
    step="mid",
    color="tab:orange",
    alpha=0.40,
)


plt.axvline(
    all_branches_flat_below_keV,
    color="tab:orange",
    linestyle=":",
    linewidth=1.5,
    label="Flat estimate for all branches ends",
)

plt.axvline(
    ground_branch_flat_below_keV,
    color="tab:green",
    linestyle=":",
    linewidth=1.5,
    label="Flat estimate for ground branch ends",
)

plt.title("Detector 1: approximate branching-ratio estimate")
plt.xlabel("Reaction product energy [keV]")
plt.ylabel("Counts")
plt.yscale("log")
plt.legend(loc="upper right", fontsize=8)
plt.show()